# G1 Academy 1 - Participant: DDS, speech, and lights


## Introduction
This module teaches the DDS building blocks used everywhere else. Low state, odom, and SLAM use subscribers. Lowcmd and Dex3 use exclusive publishers. Loco, ModeSwitcher, Audio, and SLAM are SDK clients. Only the Piper WAV conversion/playback helper remains in util.py.

## ChannelFactoryInitialize: one global DDS factory per kernel

DDS channel construction has global process state. Call ChannelFactoryInitialize once, before any ChannelSubscriber or ChannelPublisher is created, and always use the same domain ID and network interface for the rest of that notebook kernel. Reinitializing it with a different interface/domain can produce silent discovery failures or errors. Restart the kernel if the target interface/domain changes.

The guard below is deliberately in the notebook rather than util.py: participants must understand and control their own DDS process lifecycle. Use the same guard in later notebooks, or restart the kernel before running them independently.


## Native SDK pattern
Each notebook imports native Unitree classes directly. ChannelFactoryInitialize is called once per kernel before creating ChannelSubscriber or ChannelPublisher objects. Subscribers register a callback and cache the newest message; publishers write typed messages; SDK clients initialize once and expose request/response operations. util.py is not a robot wrapper.


## Task 1 - Build a globally safe latest-message subscriber

ChannelFactoryInitialize creates global DDS process state. Implement the guard below, call it before constructing any subscriber, then implement Latest to cache a message and its receipt time. This pattern is reused for lowstate, odometry, SLAM state, and hand state.


In [ ]:
import time
from unitree_sdk2py.core.channel import ChannelFactoryInitialize, ChannelSubscriber
from unitree_sdk2py.idl.unitree_hg.msg.dds_ import LowState_

_factory_config = None
def ensure_channel_factory(domain_id, interface):
    global _factory_config
    config = (int(domain_id), str(interface))
    # TODO: initialize once, reject a conflicting config, and return config.
    pass

ensure_channel_factory(0, "eth0")
class Latest:
    def __init__(self, topic, message_type):
        self.message=None; self.timestamp=0.0
        self.subscriber=ChannelSubscriber(topic, message_type)
        self.subscriber.Init(self.callback, 10)
    def callback(self, message):
        # TODO: cache message and time.time().
        pass
    def fresh(self, max_age_s=0.5):
        # TODO: return whether message exists and age is within max_age_s.
        pass
lowstate_sub=Latest("rt/lowstate", LowState_)


## Task 2 - Use the native AudioClient with Piper
AudioClient is the SDK request client. util.play_piper_text does only Piper synthesis, WAV conversion, and PlayStream formatting; you initialize and own the client.


In [ ]:
from unitree_sdk2py.g1.audio.g1_audio_client import AudioClient
from util import play_piper_text
audio_client = AudioClient(); audio_client.SetTimeout(5.0); audio_client.Init()
def say(text, language="en"):
    # TODO: call play_piper_text(audio_client, text, language=language).
    pass


## Task 3 - Publish/refresh headlights safely
LedControl belongs to AudioClient. Create a thread that refreshes the selected RGB color, then terminate/join it before a mode/controller transition.


In [ ]:
import threading
_light_stop = threading.Event(); _light_thread = None
def set_headlight(rgb):
    # TODO: return audio_client.LedControl(*rgb).
    pass
def start_headlight(rgb=(0, 0, 255), interval_s=0.2):
    # TODO: create one cancellable refresh thread.
    pass
def stop_headlight():
    # TODO: signal and join the thread.
    pass


### Safety
Run no command cell until the subscriber state is fresh, controller ownership is known, the space is clear, and a damp path is available. Code is not invoked automatically.
